# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "gold"
datamart = "compliance"
extrator_bronze = AlinharETL(bucket,f"{datamart}/cnae")

2024-12-14 21:56:24,112 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [3]:
compliance_flat_ibge_cnae = extrator_bronze.criar_view_temporaria('gold/compliance/dim_ibge_cnae','dim_ibge_cnae')

2024-12-14 21:56:38,425 - INFO - Aguarde... Criando View (gold/compliance/dim_ibge_cnae)
2024-12-14 21:56:42,985 - INFO - View criada com sucesso


In [5]:
compliance_flat_ibge_cnae.show()

+-------+--------------------+--------------------+-------------+--------------------+
|sk_cnae|          dsc_classe|       dsc_subclasse|dsc_industria|      dt_atualizacao|
+-------+--------------------+--------------------+-------------+--------------------+
|8591100|  Ensino De Esportes|  Ensino De Esportes|             |2024-12-14 21:48:...|
|9329801|Atividades De Rec...|Discotecas, Dance...|             |2024-12-14 21:48:...|
|5112999|Transporte Aéreo ...|Outros Serviços D...|             |2024-12-14 21:48:...|
|3701100|Gestão De Redes D...|Gestão De Redes D...|             |2024-12-14 21:48:...|
|3831999|Recuperação De Ma...|Recuperação De Ma...|             |2024-12-14 21:48:...|
|3900500|Descontaminação E...|Descontaminação E...|             |2024-12-14 21:48:...|
|4635403|Comércio Atacadis...|Comércio Atacadis...|             |2024-12-14 21:48:...|
|4651601|Comércio Atacadis...|Comércio Atacadis...|             |2024-12-14 21:48:...|
|1099602|Fabricação De Pro...|Fabricação De

# Tratamentos na tabela 

In [4]:
sql_query = """
        SELECT 
            sk_cnae,
            dsc_classe,
            dsc_subclasse,   
            dsc_industria
        FROM  compliance_flat_ibge_cnae
"""

In [5]:
compliance_flat_ibge_cnae = extrator_bronze.carregar_dados_delta(sql_query)

2024-09-25 19:19:30,450 - INFO - Aguarde... Executando Query (Delta)
2024-09-25 19:19:30,451 - INFO - 
        SELECT 
            sk_cnae,
            dsc_classe,
            dsc_subclasse,   
            dsc_industria
        FROM  compliance_flat_ibge_cnae

2024-09-25 19:19:30,511 - INFO - Query (Delta) executada com sucesso


# Gravação no datalake

In [6]:
extrator_bronze.caminho_tabela_delta = 's3a://gold/ServicoDadosIBGE/FlatServicoDadosIBGE'

In [7]:
extrator_bronze.salvar_delta(compliance_flat_ibge_cnae, 'overwrite')

2024-09-25 19:19:30,524 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 19:19:37,767 - INFO - Dados persistidos com sucesso
2024-09-25 19:19:37,768 - INFO - s3a://gold/ServicoDadosIBGE/FlatServicoDadosIBGE
2024-09-25 19:19:37,768 - INFO - Aguarde... Realizando optimize
2024-09-25 19:19:41,916 - INFO - Optimize executado com sucesso.
2024-09-25 19:19:41,916 - INFO - Aguarde... Realizando vacuum
2024-09-25 19:19:58,192 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [6]:
extrator_bronze.parar_sessao()

2024-12-14 21:58:51,624 - INFO - Sessão Spark finalizada.
